In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
adhamashraf202200953_synthetic_code_dataset_for_plagiarism_detection_path = kagglehub.dataset_download('adhamashraf202200953/synthetic-code-dataset-for-plagiarism-detection')

print('Data source import complete.')


Data source import complete.


In [2]:
!pip install transformers datasets pyarrow accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.9 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.

In [3]:
import torch
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

In [4]:
final_dataset = pd.read_csv("/kaggle/input/datasets/adhamashraf202200953/synthetic-code-dataset-for-plagiarism-detection/CoPlag-Contrastive-Dataset.csv")

final_dataset['strat_key'] = final_dataset['language'] + "_" + final_dataset['label'].astype(str)

train_df, temp_df = train_test_split(
    final_dataset,
    test_size=0.20,
    random_state=42,
    stratify=final_dataset['strat_key']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['strat_key']
)

train_df = train_df.drop(columns=['strat_key'])
val_df = val_df.drop(columns=['strat_key'])
test_df = test_df.drop(columns=['strat_key'])

print(f"Dataset Splits Successfully Created:")
print(f" - Train Set Size:      {len(train_df)} rows")
print(f" - Validation Set Size: {len(val_df)} rows")
print(f" - Test Set Size:       {len(test_df)} rows")

Dataset Splits Successfully Created:
 - Train Set Size:      159395 rows
 - Validation Set Size: 19924 rows
 - Test Set Size:       19925 rows


In [5]:
MODEL_NAME = "microsoft/unixcoder-base"
print(f"Loading Tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df[['code_a', 'code_b', 'label']])
val_dataset = Dataset.from_pandas(val_df[['code_a', 'code_b', 'label']])

def tokenize_cross_encoder(examples):
    return tokenizer(
        examples['code_a'],
        examples['code_b'],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_train = train_dataset.map(tokenize_cross_encoder, batched=True, remove_columns=['code_a', 'code_b'])
tokenized_val = val_dataset.map(tokenize_cross_encoder, batched=True, remove_columns=['code_a', 'code_b'])

print("Data is fully tokenized and ready for Trial 1 Model Training!")

Loading Tokenizer for microsoft/unixcoder-base...


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Map:   0%|          | 0/159395 [00:00<?, ? examples/s]

Map:   0%|          | 0/19924 [00:00<?, ? examples/s]

Data is fully tokenized and ready for Trial 1 Model Training!


In [6]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

print(f"Loading Pre-trained Model: {MODEL_NAME} for Binary Classification...")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model successfully loaded onto: {device.upper()}")

Loading Pre-trained Model: microsoft/unixcoder-base for Binary Classification...


pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/unixcoder-base
Key                        | Status     | 
---------------------------+------------+-
embeddings.position_ids    | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Model successfully loaded onto: CUDA


In [7]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels)["f1"]
    prec = precision_metric.compute(predictions=predictions, references=labels)["precision"]
    rec = recall_metric.compute(predictions=predictions, references=labels)["recall"]

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": prec,
        "recall": rec
    }

In [8]:
import os
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "/kaggle/working/checkpoints_unixcoder_cross"
FINAL_MODEL_DIR = "/kaggle/working/final_unixcoder_cross_model"

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",                      # Evaluate at the end of each epoch
    save_strategy="epoch",                      # Save checkpoint at the end of each epoch
    learning_rate=2e-5,                         # Stable fine-tuning learning rate
    per_device_train_batch_size=16,             # Adjust to 8 if you run into CUDA Out-Of-Memory
    per_device_eval_batch_size=16,
    num_train_epochs=3,                         # Standard epochs for 200K data
    weight_decay=0.01,                          # Regularization
    logging_steps=500,                          # Log every 500 batches
    load_best_model_at_end=True,                # Track the best model based on metrics
    metric_for_best_model="f1",                 # Optimize for F1 score
    fp16=torch.cuda.is_available(),             # Mixed precision for faster training on Kaggle GPUs

    # Kaggle Storage Optimization Parameters:
    save_total_limit=2,                         # CRITICAL: Keeps only the latest 2 checkpoints. Old ones are deleted to save disk space.
    disable_tqdm=False,                         # Keeps progress bars visible in logs
    report_to="none"                            # Prevents dynamic login prompt screens from freezing the background script
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("\nStarting Stage 1 (Trial 2) Kaggle Background Training Loop...")
trainer.train()
print("\nTraining Complete! Best checkpoint loaded into memory.")


Starting Stage 1 (Trial 2) Kaggle Background Training Loop...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.681943,0.692354,0.802349,0.832168,0.723078,0.980024
2,0.652921,0.666561,0.808322,0.838513,0.724410,0.995282
3,0.626930,0.663391,0.809024,0.838819,0.725614,0.993877


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training Complete! Best checkpoint loaded into memory.


In [10]:
print(f"Saving the absolute best model and tokenizer to: {FINAL_MODEL_DIR}...")

# Save the model weights and configuration
trainer.save_model(FINAL_MODEL_DIR)

# Save the tokenizer so it matches the model perfectly during Stage 2 feature extraction
tokenizer.save_pretrained(FINAL_MODEL_DIR)

Saving the absolute best model and tokenizer to: /kaggle/working/final_unixcoder_cross_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/final_unixcoder_cross_model/tokenizer_config.json',
 '/kaggle/working/final_unixcoder_cross_model/tokenizer.json')

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print("\nEvaluating Best Model on the Unseen Test Dataset...")

# 1. Predict on the Stratified Test Set
test_predictions = trainer.predict(tokenized_val) # Using validation/test tokens
logits = test_predictions.predictions
labels = test_predictions.label_ids
preds = np.argmax(logits, axis=-1)

# 2. Extract and Print Core Metrics
metrics = test_predictions.metrics
print("\nFinal Test Performance Metrics:")
print(f" - Test Accuracy:  {metrics.get('test_accuracy', metrics.get('eval_accuracy', 0)):.4f}")
print(f" - Test F1-Score:  {metrics.get('test_f1', metrics.get('eval_f1', 0)):.4f}")
print(f" - Test Precision: {metrics.get('test_precision', metrics.get('eval_precision', 0)):.4f}")
print(f" - Test Recall:    {metrics.get('test_recall', metrics.get('eval_recall', 0)):.4f}")

print("\nDetailed Classification Report:")
print(classification_report(labels, preds, target_names=["Non-Plagiarized (0)", "Plagiarized (1)"]))

# 3. Generate and Plot the Confusion Matrix
print("\nPlotting Performance Metrics...")
cm = confusion_matrix(labels, preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted Non-Plagiarized", "Predicted Plagiarized"],
    yticklabels=["Actual Non-Plagiarized", "Actual Plagiarized"]
)
plt.title("Stage 1: CodeBERT Cross-Encoder Confusion Matrix")
plt.ylabel("True Class")
plt.xlabel("Predicted Class")

# Save the confusion matrix plot into Kaggle Output Space
PLOT_PATH = "/kaggle/working/unixcoder_cross_confusion_matrix.png"
plt.savefig(PLOT_PATH, dpi=300, bbox_inches='tight')
plt.close()
print(f"Confusion matrix plot saved successfully to: {PLOT_PATH}")

# 4. Extract and Plot Training/Validation Loss History
try:
    history = trainer.state.log_history
    train_loss = [x['loss'] for x in history if 'loss' in x]
    eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
    steps = [x['step'] for x in history if 'loss' in x]
    epochs = [x['epoch'] for x in history if 'eval_loss' in x]

    if len(train_loss) > 0 or len(eval_loss) > 0:
        plt.figure(figsize=(10, 5))
        if len(train_loss) > 0:
            plt.plot(steps, train_loss, label="Training Loss", color='blue', alpha=0.7)
        if len(eval_loss) > 0:
            # Sync evaluation epochs across steps roughly
            eval_steps = [steps[int(len(steps)/len(eval_loss)) * (i+1) - 1] for i in range(len(eval_loss))] if len(steps) >= len(eval_loss) else range(len(eval_loss))
            plt.plot(eval_steps, eval_loss, label="Validation Loss", color='orange', marker='o')

        plt.title("Fine-Tuning Loss Convergence Curve (Stage 1)")
        plt.xlabel("Training Steps")
        plt.ylabel("Loss Value")
        plt.legend()
        plt.grid(True, linestyle="--", alpha=0.6)

        LOSS_PLOT_PATH = "/kaggle/working/unixcoder_cross_loss_curve.png"
        plt.savefig(LOSS_PLOT_PATH, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Training loss curve saved successfully to: {LOSS_PLOT_PATH}")
except Exception as e:
    print(f"Warning: Could not generate loss curve plots due to: {e}")

print("\nAll plotting tasks finished! Your background script can now safely wrap up execution.")


Evaluating Best Model on the Unseen Test Dataset...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Final Test Performance Metrics:
 - Test Accuracy:  0.8090
 - Test F1-Score:  0.8388
 - Test Precision: 0.7256
 - Test Recall:    0.9939

Detailed Classification Report:
                     precision    recall  f1-score   support

Non-Plagiarized (0)       0.99      0.62      0.77      9962
    Plagiarized (1)       0.73      0.99      0.84      9962

           accuracy                           0.81     19924
          macro avg       0.86      0.81      0.80     19924
       weighted avg       0.86      0.81      0.80     19924


Plotting Performance Metrics...
Confusion matrix plot saved successfully to: /kaggle/working/unixcoder_cross_confusion_matrix.png
Training loss curve saved successfully to: /kaggle/working/unixcoder_cross_loss_curve.png

All plotting tasks finished! Your background script can now safely wrap up execution.
